In [15]:
// ==== Setup ====

import java.util.*;

// wall-clock time of r, in milliseconds
long millis(Runnable r) {
    System.gc();            // start from a clean heap: less noise
    long start = System.nanoTime();
    r.run();
    return (System.nanoTime() - start) / 1_000_000;
}

# Stacks: recap

- **LIFO**: `push`, `pop`, `peek`, all at the **top**.
- Array-based (top at `size - 1`) or linked (top at the first node): `O(1)` per operation.

## Reading a stack

- The ADT restricts how a stack **changes**: only at the top.
- In practice, **read-only** iteration is usually allowed: from the top to the bottom, without changing the stack.

# Queues

## Queue ADT

- Elements are added at one end (**back**), removed at the other (**front**).
- **FIFO**: first in, first out.
- Examples: a line at a counter, jobs sent to a printer, requests to a server.

| operation | |
|---|---|
| `enqueue(e)` | add `e` at the back |
| `dequeue()` | remove and return the front element |
| `peek()` | return the front element, without removing it |
| `isEmpty()`, `size()` | |

In [16]:
interface MyQueue<E> {
    void enqueue(E e);

    // throw NoSuchElementException if the queue is empty
    E dequeue();
    E peek();

    boolean isEmpty();
    int size();
}

## Linked queue

- Additions and removals at **different** ends.
- Singly linked, with `head` and `tail`:

| front at | `enqueue` | `dequeue` |
|---|---|---|
| first node | ? | ? |
| last node | ? | ? |

Front at the first node:
- `enqueue`: `add` at the tail
- `dequeue`: `removeFirst`
- Fields: `front`, `back`, `size`

In [17]:
class LinkedQueue<E> implements MyQueue<E> {
    private static class Node<E> {
        E data;
        Node<E> next;       // the node behind this one

        Node(E data) { this.data = data; }
    }

    private Node<E> front;  // null if empty
    private Node<E> back;   // null if empty
    private int size;

    public int size() { return size; }
    public boolean isEmpty() { return front == null; }

    public void enqueue(E e) {
        Node<E> n = new Node<>(e);
        if (back == null) front = n;    // the queue was empty
        else back.next = n;
        back = n;
        size++;
    }

    public E dequeue() {
        if (front == null) throw new NoSuchElementException("empty queue");
        E e = front.data;
        front = front.next;
        if (front == null) back = null; // removed the only node
        size--;
        return e;
    }

    public E peek() {
        if (front == null) throw new NoSuchElementException("empty queue");
        return front.data;
    }

    // front to back
    @Override
    public String toString() {
        StringBuilder sb = new StringBuilder("[");
        for (Node<E> n = front; n != null; n = n.next) {
            sb.append(n.data);
            if (n.next != null) sb.append(", ");
        }
        return sb.append("]").toString();
    }
}

In [18]:
MyQueue<String> q = new LinkedQueue<>();
q.enqueue("ann");
q.enqueue("bob");
q.enqueue("cat");
System.out.println(q + "  peek=" + q.peek());
System.out.println(q.dequeue() + " " + q.dequeue() + " " + q);

q.dequeue();                    // now empty
q.enqueue("dan");
System.out.println(q + "  size=" + q.size());

[ann, bob, cat]  peek=ann
ann bob [cat]
[dan]  size=1


## Array-based queue

### First attempt: front at index `0`
- `enqueue`: `data[size]`, `O(1)` amortized
- `dequeue`: `data[0]`, then shift everything left: `O(N)`

### Second attempt: a moving front
- `front`: index of the first element. `dequeue`: `front++`, no shift.
- Elements in `data[front..front+size-1]`.
- `enqueue` 8 and `dequeue` 6 times, capacity 8: what is wrong?

## Circular array

- After the last cell, continue from cell `0`.
- Element `i` (from the front): `data[(front + i) % capacity]`
- Next free cell: `data[(front + size) % capacity]`
- `dequeue`: `front = (front + 1) % capacity`
- Full only when `size == capacity`.

### Growing

- `Arrays.copyOf(data, 2 * capacity)` keeps the cells in the same positions. Why does it not work here?
- Copy element `i` from the front into cell `i`, then `front = 0`.

In [19]:
class ArrayQueue<E> implements MyQueue<E> {
    private E[] data;
    private int front;      // index of the first element
    private int size;       // elements in data[front], data[front+1], ... wrapping around

    @SuppressWarnings("unchecked")
    ArrayQueue() { data = (E[]) new Object[4]; }

    public int size() { return size; }
    public boolean isEmpty() { return size == 0; }

    @SuppressWarnings("unchecked")
    private void grow() {
        E[] bigger = (E[]) new Object[2 * data.length];
        for (int i = 0; i < size; i++)
            bigger[i] = data[(front + i) % data.length];
        data = bigger;
        front = 0;
    }

    public void enqueue(E e) {
        if (size == data.length) grow();
        data[(front + size) % data.length] = e;
        size++;
    }

    public E dequeue() {
        if (size == 0) throw new NoSuchElementException("empty queue");
        E e = data[front];
        data[front] = null;
        front = (front + 1) % data.length;
        size--;
        return e;
    }

    public E peek() {
        if (size == 0) throw new NoSuchElementException("empty queue");
        return data[front];
    }

    // front to back
    @Override
    public String toString() {
        StringBuilder sb = new StringBuilder("[");
        for (int i = 0; i < size; i++) {
            sb.append(data[(front + i) % data.length]);
            if (i < size - 1) sb.append(", ");
        }
        return sb.append("]").toString();
    }

    // the array as it is, to see the wrap-around
    public String internals() {
        return Arrays.toString(data) + "  front=" + front + "  size=" + size;
    }
}

In [20]:
ArrayQueue<Integer> aq = new ArrayQueue<>();
for (int i = 1; i <= 4; i++) aq.enqueue(i);
System.out.println(aq + "   " + aq.internals());

aq.dequeue();
aq.dequeue();
System.out.println(aq + "   " + aq.internals());

aq.enqueue(5);                  // wraps around
aq.enqueue(6);
System.out.println(aq + "   " + aq.internals());

aq.enqueue(7);                  // full: grows
System.out.println(aq + "   " + aq.internals());

[1, 2, 3, 4]   [1, 2, 3, 4]  front=0  size=4
[3, 4]   [null, null, 3, 4]  front=2  size=2
[3, 4, 5, 6]   [5, 6, 3, 4]  front=2  size=4
[3, 4, 5, 6, 7]   [3, 4, 5, 6, 7, null, null, null]  front=0  size=5


- Cost of `enqueue`, `dequeue`, `peek`?
- Only `front` and `size` are fields. Where is the back?

## Array vs. linked queue

| | array (circular) | linked |
|---|---|---|
| `enqueue` | `O(1)` amortized, `O(N)` worst case | `O(1)` |
| `dequeue`, `peek` | `O(1)` | `O(1)` |
| memory | one cell per element, plus unused capacity | one node per element |

In [28]:
System.out.printf("%10s %10s %10s%n", "N", "array ms", "linked ms");
for (int n = 1_000_000; n <= 8_000_000; n *= 2) {
    final int N = n;
    long arr = millis(() -> {
        MyQueue<Integer> q = new ArrayQueue<>();
        for (int i = 0; i < N; i++) q.enqueue(i % 10);    // small Integers are cached: no allocation
        while (!q.isEmpty()) q.dequeue();
    });
    long lnk = millis(() -> {
        MyQueue<Integer> q = new LinkedQueue<>();
        for (int i = 0; i < N; i++) q.enqueue(i % 10);
        while (!q.isEmpty()) q.dequeue();
    });
    System.out.printf("%,10d %10d %10d%n", N, arr, lnk);
}

         N   array ms  linked ms
 1,000,000         63        127
 2,000,000         72        147
 4,000,000        118        172
 8,000,000        211        409


- Compare with the stack timings: same picture?
- Which one would you choose? When would the other one be better?

## A list on a circular array

`MyArrayListDouble`, plus a `front` field: element `i` is in `data[(front + i) % capacity]`.

| | `MyArrayListDouble` | circular array |
|---|---|---|
| `get(i)`, `set(i, e)` | `O(1)` | `O(1)` |
| `add`, `remove` at the front | `O(N)` | `O(1)` amortized |
| `add`, `remove` at the back | `O(1)` amortized | `O(1)` amortized |
| `add`, `remove` in the middle | `O(N)` | `O(N)` |

- Both ends in `O(1)`, and still access by position in `O(1)`.
- In the middle: shift toward the **nearer** end, at most `N/2` moves.
- The price: one `+` and one `%` per access.

### Does `java.util.ArrayList` do this?

- No: always starts at index `0`, no `front` field.
- `get`/`set` is its hot path: no extra `+` and `%` there.
- The circular-array trick is used instead by `java.util.ArrayDeque`.